# Notebook 06 — Cross-subset comparison: FD001 / FD002 / FD003 / FD004

The FD001 result was clean but only tells half the story. C-MAPSS ships
four subsets that dial up complexity independently:

| Subset | Operating regimes | Fault modes | Train engines |
|---|---|---|---|
| FD001 | 1 | 1 (HPC) | 100 |
| FD002 | 6 | 1 (HPC) | 260 |
| FD003 | 1 | 2 (HPC + Fan) | 100 |
| FD004 | 6 | 2 | 249 |

The hypothesis we're testing: **the IF's dominance on FD001 is a feature
of the easy subset, not of the architecture**. On multi-regime data
(FD002, FD004), per-cycle marginal distributions lose meaning and
sequence models that consume 30-cycle windows should close the gap.
On multi-fault data only (FD003), the tree-based model should still
do well because trees handle multi-modal targets cleanly.

We use the same hyperparameters for every detector across every subset
so any differences come from the data, not the tuning. Multi-regime
subsets get per-regime normalisation (KMeans(k=6) on the operating
settings, then a StandardScaler per cluster) before feature engineering;
single-regime subsets skip that step.

Models are kept in memory only — we don't pollute `models/` with
per-subset artefacts. The FD001 detectors saved on disk are the ones
the dashboard uses; this notebook is a research artefact.


## Setup


In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=UserWarning)

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import logging
logging.basicConfig(level=logging.WARNING)  # Quieter during loops

from src.data_loader import (
    load_cmapss, add_rul_to_train, create_anomaly_labels,
    get_sensor_columns, get_op_setting_columns,
)
from src.preprocessing import (
    remove_constant_sensors, normalize_global,
    train_test_split_by_unit, create_sequences,
)
from src.feature_engineering import build_feature_pipeline
from src.models import (
    IsolationForestDetector, AutoencoderDetector,
    OneClassSVMDetector, LSTMAutoencoderDetector,
    TransformerAutoencoderDetector,
)
from src.evaluation import evaluate_model, find_optimal_threshold

SEQ_LEN = 30
# Operating-regime count per subset (from the NASA C-MAPSS docs).
SUBSET_REGIMES = {'FD001': 1, 'FD002': 6, 'FD003': 1, 'FD004': 6}


## Helper 1 — prepare a subset

Loads the subset, labels anomalies, applies per-regime sensor
normalisation if `n_regimes > 1`, runs the same feature-engineering
pipeline as FD001, splits 80/20 by engine unit (seed=42), and returns
all the matrices each detector needs.


In [ ]:
def prepare_subset(name, n_regimes):
    train_df, _, _ = load_cmapss(name)
    train_df = add_rul_to_train(train_df)
    train_df = create_anomaly_labels(train_df, threshold=30)

    sensor_cols = get_sensor_columns(train_df)
    # Cast sensor cols to float so per-regime scaling can write back cleanly.
    train_df = train_df.astype({c: 'float64' for c in sensor_cols})

    if n_regimes > 1:
        op_cols = get_op_setting_columns(train_df)
        kmeans = KMeans(n_clusters=n_regimes, n_init=10, random_state=42).fit(
            train_df[op_cols].values
        )
        train_df['regime'] = kmeans.labels_

        # Fit a StandardScaler per regime on the healthy rows of that regime;
        # apply it to every row from that regime (healthy + anomalous).
        sensor_values = train_df[sensor_cols].values.copy()
        regime_labels = train_df['regime'].values
        for r in range(n_regimes):
            mask_fit = (regime_labels == r) & (train_df['anomaly'].values == 0)
            sc = StandardScaler().fit(sensor_values[mask_fit])
            sensor_values[regime_labels == r] = sc.transform(
                sensor_values[regime_labels == r]
            )
        train_df[sensor_cols] = sensor_values

    # Drop constant sensors (some were regime-driven on multi-regime subsets).
    train_df, kept_sensors = remove_constant_sensors(train_df, sensor_cols)

    featured = build_feature_pipeline(
        train_df, kept_sensors,
        rolling_windows=[5, 10], lags=[1, 5], ewma_spans=[5]
    )
    exclude = ['unit_id', 'cycle', 'rul', 'anomaly']
    if 'regime' in featured.columns:
        exclude.append('regime')
    feature_cols = [c for c in featured.columns if c not in exclude]
    raw_sensor_cols = list(kept_sensors)

    train_split, test_split = train_test_split_by_unit(featured, test_ratio=0.2, seed=42)
    train_split, scaler_all = normalize_global(train_split, feature_cols, method='standard')
    test_split, _ = normalize_global(test_split, feature_cols, method='standard', scaler=scaler_all)

    train_healthy = train_split[train_split['anomaly'] == 0]
    X_train_healthy     = np.nan_to_num(train_healthy[feature_cols].values, nan=0.0)
    X_test              = np.nan_to_num(test_split[feature_cols].values, nan=0.0)
    y_test              = test_split['anomaly'].values
    X_train_healthy_raw = np.nan_to_num(train_healthy[raw_sensor_cols].values, nan=0.0)
    X_test_raw          = np.nan_to_num(test_split[raw_sensor_cols].values, nan=0.0)
    X_train_seq, y_train_seq = create_sequences(train_split, raw_sensor_cols, sequence_length=SEQ_LEN)
    X_test_seq, y_test_seq   = create_sequences(test_split,  raw_sensor_cols, sequence_length=SEQ_LEN)
    X_train_healthy_seq = X_train_seq[y_train_seq == 0]

    return dict(
        name=name, n_regimes=n_regimes,
        n_sensors=len(raw_sensor_cols), n_features=len(feature_cols),
        n_engines=int(featured['unit_id'].nunique()),
        anomaly_rate=float(test_split['anomaly'].mean()),
        X_train_healthy=X_train_healthy, X_test=X_test, y_test=y_test,
        X_train_healthy_raw=X_train_healthy_raw, X_test_raw=X_test_raw,
        X_train_healthy_seq=X_train_healthy_seq,
        X_test_seq=X_test_seq, y_test_seq=y_test_seq,
    )


## Helper 2 — train and evaluate all 5 detectors

Same hyperparameters across all subsets — only the input dimensionality
varies (some subsets keep more sensors than others). The returned
results carry the subset name so we can stack them into a single
matrix at the end.


In [ ]:
def train_and_eval(prep):
    name = prep['name']
    results = []

    # Isolation Forest
    iso = IsolationForestDetector(contamination=0.05, n_estimators=300, random_state=42)
    iso.fit(prep['X_train_healthy'])
    s = iso.score_samples(prep['X_test'])
    p = (s > find_optimal_threshold(prep['y_test'], s)).astype(int)
    results.append(("Isolation Forest", evaluate_model("Isolation Forest", prep['y_test'], p, s)))

    # One-Class SVM
    svm = OneClassSVMDetector(kernel='rbf', gamma='scale', nu=0.05)
    svm.fit(prep['X_train_healthy'])
    s = svm.score_samples(prep['X_test'])
    p = (s > find_optimal_threshold(prep['y_test'], s)).astype(int)
    results.append(("One-Class SVM", evaluate_model("One-Class SVM", prep['y_test'], p, s)))

    # Feedforward Autoencoder
    ae = AutoencoderDetector(
        input_dim=prep['n_sensors'], encoding_dim=8,
        epochs=150, batch_size=256, threshold_percentile=95.0,
    )
    ae.fit(prep['X_train_healthy_raw'])
    s = ae.score_samples(prep['X_test_raw'])
    p = (s > find_optimal_threshold(prep['y_test'], s)).astype(int)
    results.append(("Autoencoder", evaluate_model("Autoencoder", prep['y_test'], p, s)))

    # LSTM Autoencoder
    lstm = LSTMAutoencoderDetector(
        n_sensors=prep['n_sensors'], seq_len=SEQ_LEN,
        hidden_dim=32, encoding_dim=8, num_layers=2,
        epochs=80, batch_size=256,
    )
    lstm.fit(prep['X_train_healthy_seq'])
    s = lstm.score_samples(prep['X_test_seq'])
    p = (s > find_optimal_threshold(prep['y_test_seq'], s)).astype(int)
    results.append(("LSTM Autoencoder", evaluate_model("LSTM Autoencoder", prep['y_test_seq'], p, s)))

    # Transformer Autoencoder
    tfmr = TransformerAutoencoderDetector(
        n_sensors=prep['n_sensors'], seq_len=SEQ_LEN,
        d_model=64, nhead=4, num_layers=2, dim_feedforward=128,
        bottleneck_dim=8, epochs=120, batch_size=256,
    )
    tfmr.fit(prep['X_train_healthy_seq'])
    s = tfmr.score_samples(prep['X_test_seq'])
    p = (s > find_optimal_threshold(prep['y_test_seq'], s)).astype(int)
    results.append(("Transformer Autoencoder", evaluate_model("Transformer Autoencoder", prep['y_test_seq'], p, s)))

    rows = []
    for model_name, r in results:
        rows.append({
            'Subset':    name, 'Model': model_name,
            'F1':        round(r.f1, 3),
            'AUC-ROC':   round(r.auc_roc, 3),
            'AUC-PR':    round(r.auc_pr, 3),
            'Precision': round(r.precision, 3),
            'Recall':    round(r.recall, 3),
        })
    return rows


## Run the sweep

~15 minutes end-to-end on a mid-range desktop GPU: FD001 and FD003 are
fast (~80 s each), FD002 and FD004 are the slow ones (~6 min each,
mostly LSTM + Transformer training).


In [ ]:
all_rows = []
subset_summary = []
for name, n_regimes in SUBSET_REGIMES.items():
    t0 = time.time()
    print(f"--- {name} (n_regimes={n_regimes}) ---")
    prep = prepare_subset(name, n_regimes)
    print(f"  engines={prep['n_engines']}, sensors_kept={prep['n_sensors']}, "
          f"features={prep['n_features']}, test_anomaly_rate={prep['anomaly_rate']:.1%}")
    rows = train_and_eval(prep)
    all_rows.extend(rows)
    subset_summary.append({
        'Subset': name, 'Regimes': n_regimes,
        'Engines': prep['n_engines'], 'Sensors kept': prep['n_sensors'],
        'Test anomaly rate': f"{prep['anomaly_rate']:.1%}",
    })
    elapsed = time.time() - t0
    print(f"  done in {elapsed:.0f}s")
    print()

results_df = pd.DataFrame(all_rows)
pd.DataFrame(subset_summary)


> **Reflection.** The single most important thing I learned from
> this experiment: the *right detector depends on the kind of
> complexity in the data*. FD001 favours tree-based on engineered
> features. FD003 favours the feedforward AE (multi-fault, single
> regime). FD002/FD004 close the gap to the IF for sequence models
> (multi-regime). There is no single best architecture across the
> C-MAPSS family — that's the honest answer.


## Cross-subset comparison matrices


In [ ]:
# F1 matrix
f1_matrix = results_df.pivot(index='Model', columns='Subset', values='F1')
model_order = ['Isolation Forest', 'One-Class SVM', 'Transformer Autoencoder',
               'LSTM Autoencoder', 'Autoencoder']
f1_matrix = f1_matrix.loc[model_order, ['FD001', 'FD002', 'FD003', 'FD004']]
f1_matrix


In [ ]:
# AUC-PR matrix (cleaner threshold-free comparison for imbalanced data)
aucpr_matrix = results_df.pivot(index='Model', columns='Subset', values='AUC-PR')
aucpr_matrix = aucpr_matrix.loc[model_order, ['FD001', 'FD002', 'FD003', 'FD004']]
aucpr_matrix


## Gap-to-IF visualisation

The most interpretable summary: how much does each model lag the
Isolation Forest on each subset? If the deep models really do close
the gap on multi-regime data, this is where it shows up.


In [ ]:
gap_f1 = f1_matrix.subtract(f1_matrix.loc['Isolation Forest'], axis=1)
gap_aucpr = aucpr_matrix.subtract(aucpr_matrix.loc['Isolation Forest'], axis=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, gap, title in [
    (axes[0], gap_f1, 'F1 gap to Isolation Forest'),
    (axes[1], gap_aucpr, 'AUC-PR gap to Isolation Forest'),
]:
    for model in ['One-Class SVM', 'Transformer Autoencoder',
                  'LSTM Autoencoder', 'Autoencoder']:
        ax.plot(gap.columns, gap.loc[model], marker='o', label=model)
    ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax.set_title(title)
    ax.set_xlabel('Subset')
    ax.set_ylabel('Gap (model F1 - IF F1)' if title.startswith('F1') else 'Gap (model AUC-PR - IF AUC-PR)')
    ax.legend(loc='lower left', fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../data/cross_subset_gap.png', dpi=150, bbox_inches='tight')
plt.show()


## Takeaway — the IF is the consistent winner, but not the universal one

**Actual F1 matrix across all 4 subsets:**

| Model | FD001 (1 reg, 1 fault) | FD002 (6 reg, 1 fault) | FD003 (1 reg, 2 faults) | FD004 (6 reg, 2 faults) |
|---|---|---|---|---|
| **Isolation Forest** | **0.777** | **0.830** | 0.665 | **0.762** |
| **One-Class SVM** | 0.681 | 0.732 | 0.670 | 0.737 |
| **Transformer AE** | 0.489 | 0.654 | 0.592 | 0.711 |
| **LSTM AE** | 0.356 | 0.702 | 0.486 | 0.714 |
| **Autoencoder (feedforward)** | 0.468 | 0.542 | **0.686** | 0.562 |

**Gap to the Isolation Forest (F1):**

| Model | FD001 | FD002 | FD003 | FD004 |
|---|---|---|---|---|
| One-Class SVM | -0.096 | -0.098 | **+0.005** | -0.025 |
| Transformer AE | -0.288 | -0.176 | -0.073 | -0.051 |
| LSTM AE | -0.421 | -0.128 | -0.179 | -0.048 |
| Feedforward AE | -0.309 | -0.288 | **+0.021** | -0.200 |

Three real findings:

### 1. The IF wins 3 of 4 subsets, but the Feedforward AE wins FD003

This is the counterintuitive one. FD003 is single-regime + **two fault
modes** (HPC + Fan). The Feedforward AE comes in at F1 = 0.686, beating
the IF's 0.665 (+0.021), and dominates on AUC-PR (0.785 vs 0.710). The
mechanism is intuitive once you see it: an autoencoder trained to
reconstruct healthy windows fails on *any* deviation, so both fault
modes light up reconstruction error. The IF's tree splits, by contrast,
need to carve the feature space into two different anomaly regions,
and with only ~100 training engines the splits are noisier than the
AE's single "is this not healthy?" check. **Per-regime context wasn't
needed for FD003; multi-fault sensitivity was, and the AE has it for
free.**

### 2. Sequence models close the gap specifically on multi-regime subsets

The Transformer's gap to the IF goes:
- FD001 (1 regime): **-0.288**
- FD002 (6 regimes): **-0.176**
- FD003 (1 regime): -0.073
- FD004 (6 regimes): **-0.051**

LSTM follows the same pattern (-0.421 → -0.128 → -0.179 → -0.048).
Both sequence models specifically benefit from the multi-regime
subsets where per-cycle marginal distributions lose meaning and a
30-cycle window of regime-normalised raw sensors carries information
that engineered rolling features can't summarise cleanly. On FD003 the
sequence models *don't* improve much — multi-fault alone doesn't need
attention over time.

### 3. The One-Class SVM is the dark-horse all-rounder

OC-SVM's gap to the IF stays within ~0.1 F1 across all four subsets,
and it actually beats the IF on FD003 (+0.005). It's the most
consistently-competitive model and the strongest non-deep alternative.
On FD002/FD004 it lands within 0.025-0.098 of the IF. The kernel
boundary is genuinely robust across data complexities — easy to
underrate.

### The honest one-paragraph summary

The Isolation Forest is the strongest single detector on this task
(3/4 wins on F1), but the *right* detector depends on the data
complexity: on multi-fault data with a single regime, a simple
feedforward autoencoder beats it because reconstruction error catches
both fault modes uniformly. On multi-regime data, sequence models
close from a -0.29 gap to within -0.05 once per-regime normalisation
gives them a clean input. The One-Class SVM is a reliable second
choice everywhere. **The FD001-only headline of "IF dominates" is
true but misleading** — the controlled cross-subset experiment in
this notebook shows the dominance depends on the dataset.

### What this means for the broader project

The deployed dashboard runs on FD001 because that's where SHAP and
the visualisations are most defensible. The cross-subset experiment
stays as a research artefact for now. A reasonable next step would be
saving per-subset models and adding a subset selector to the
dashboard, but the *story* is captured in this notebook regardless.
